# Download

Download all the files from  
https://figshare.com/articles/dataset/X-CRISP_Domain-Adaptable_and_Interpretable_CRISPR_Repair_Outcome_Prediction_-_Preprocessed_CRISPR_repair_outcomes_and_train_test_target_splits/29260037

Place them in CROP/data/XCRISP

Unzip all .zip files

An example path after unzipping would be   
CROP\data\XCRISP\U2OS\0_0_0_0_AAATATCTTTAACCTAAAAC_indels.tij.sorted.tsv


# Versions

In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("Matplotlib version:", matplotlib.__version__)

Python version: 3.11.13 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:03:15) [MSC v.1929 64 bit (AMD64)]
Pandas version: 2.3.0
Matplotlib version: 3.10.1


# Preprocess

In [2]:
MIN_DELTA = -100
MAX_DELTA = 100
NUM_CLASSES = MAX_DELTA - MIN_DELTA + 1      # 201
DELTA_OFFSET = -MIN_DELTA      

In [3]:
TSV_DIR = "../data/XCRISP/mESC_inDelphi"
OUTPUT_CSV = "../data/XCRISP_inDelphi_mESC.csv"


############################################
# FASTA LOADING
############################################

def load_fasta_id_to_seq(path):
    """
    Parses FASTA where each sequence is exactly 1 line and
    >ID OTHER INFO
    SEQUENCE
    """
    id_to_seq = {}
    with open(path, "r") as f:
        current_id = None
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith(">"):
                current_id = line[1:].split()[0]
            else:
                if current_id is None:
                    raise ValueError("Sequence found before header!")
                id_to_seq[current_id] = line
                current_id = None
    return id_to_seq


############################################
# TARGET + PAM DETECTION
############################################

def extract_target_from_id(ID):
    """
    ID looks like: 1_0_2_4_TAGTAACTCAACCTCAGCAG
    Last token after '_' is the target sequence.
    """
    return ID.split("_")[-1]


def find_pam_index(full_seq, target):
    """
    PAM is immediately AFTER the 20bp target.
    Returns index of FIRST PAM base.
    """
    pos = full_seq.find(target)
    if pos == -1:
        raise ValueError(f"Target {target} NOT found in sequence!")
    return pos + len(target)      # PAM begins right after target


############################################
# LOAD REPAIR TSV → 201-length y-vector
############################################

def load_repair_outcome_tsv(tsv_path):
    """
    Loads a single *.tij.sorted.tsv file and returns a 
    vector of length 201 (deltas -100..100).
    """

    df = pd.read_csv(tsv_path, sep="\t")

    # Validate Type column
    if "Type" not in df.columns:
        raise ValueError(f"{tsv_path} missing 'Type' column!")

    allowed = {"INSERTION", "DELETION"}
    bad = df[~df["Type"].isin(allowed)]
    if len(bad) > 0:
        print("Invalid rows in", tsv_path)
        print(bad)
        raise ValueError("Invalid Type encountered.")

    # Compute delta values
    df["length_change"] = df.apply(
        lambda row: row["Size"] if row["Type"] == "INSERTION" else -row["Size"],
        axis=1
    )

    # Filter deltas outside training range
    df = df[(df["length_change"] >= MIN_DELTA) & (df["length_change"] <= MAX_DELTA)]

    # Sum counts per delta
    grouped = df.groupby("length_change")["countEvents"].sum()

    # Build zero vector
    vec = np.zeros(NUM_CLASSES, dtype=np.float32)
    for delta, count in grouped.items():
        vec[delta + DELTA_OFFSET] = count


    return vec


############################################
# LOAD FASTA (TRAIN + TEST)
############################################

id_to_seq_1 = load_fasta_id_to_seq("../data/XCRISP/inDelphi_train.fasta")
id_to_seq_2 = load_fasta_id_to_seq("../data/XCRISP/inDelphi_test.fasta")

id_to_seq = {**id_to_seq_1, **id_to_seq_2}

print("Total FASTA entries loaded:", len(id_to_seq))


############################################
# BUILD FINAL CSV
############################################

rows = []

for ID, seq in id_to_seq.items():

    # TSV path for this target
    tsv_path = os.path.join(TSV_DIR, f"{ID}_indels.tij.sorted.tsv")

    if not os.path.exists(tsv_path):
        print(f"⚠️  WARNING: Missing TSV for ID {ID}")
        continue

    # Load repair outcomes
    y_vec = load_repair_outcome_tsv(tsv_path)

    # Extract target and PAM position
    target = extract_target_from_id(ID)
    pam_index = find_pam_index(seq, target)

    # Build row dictionary
    row = {
        "id": ID,
        "sequence": seq,
        "pam_index": pam_index,
    }

    # Add all outcomes
    for delta in range(MIN_DELTA, MAX_DELTA + 1):
        row[delta] = y_vec[delta + DELTA_OFFSET]

    rows.append(row)

# Convert to DataFrame
df_final = pd.DataFrame(rows)
print("Final dataframe shape:", df_final.shape)

# Save
df_final.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved CSV: {OUTPUT_CSV}")


Total FASTA entries loaded: 1996
⚠️  WARNING: Missing TSV for ID p53_Y163C_hg38_chr17_7675119_to_7675138_(+)_GCTTGCAGATGGCCATGGCG
⚠️  WARNING: Missing TSV for ID overbeek_spacer_84_GGCCACTGTAGTCCTCCAGG
⚠️  WARNING: Missing TSV for ID RS137854557_NF1_NM_000267.3(NF1)_ACTTACAGCTTCTTGTCTCC
⚠️  WARNING: Missing TSV for ID RS754896795_DMD_NM_004006.2(DMD)_GCTTTTCTTCAAGCTGCCCA
⚠️  WARNING: Missing TSV for ID RS121908300_GBA_NM_001005741.2(GBA)_GCCAGACACTTTGTGAAGTA
⚠️  WARNING: Missing TSV for ID RS5030804_VHL_NM_000551.3(VHL)_TGCGACTGCAGAAGATGACC
⚠️  WARNING: Missing TSV for ID RS80338732_PRPS1_NM_002764.3(PRPS1)_GCAAATACGCTATCTGTAGC
⚠️  WARNING: Missing TSV for ID 1_0_3_4_CTTACACCCCCAATCCGAGA
⚠️  WARNING: Missing TSV for ID 1_4_3_2_CCGCGGCCCTTTCCCCCTTA
⚠️  WARNING: Missing TSV for ID libcontrol_Site_A_spcas9_55bp_EMX1_chr2_72933765_to_72933787_(+)_TGCCCCTCCCTCCCTGGCCC
⚠️  WARNING: Missing TSV for ID RS199473074_SCN5A_NM_000335.4(SCN5A)_CTGATACAGTTTTCAGGGCC
⚠️  WARNING: Missing TSV for ID RS

In [4]:



# Directory with TSVs per cell type
FORECAST_DIR = "../data/XCRISP"

# Output prefix
OUTPUT_PREFIX = "../data/XCRISP_FORECasT"

############################################
# FASTA LOADING WITH PAM INDEX PARSING
############################################

def load_fasta_id_pam_seq(path):
    """
    FASTA format:
      >ID PAM_INDEX ...
      SEQUENCE
    Returns:
      dict: ID -> {"pam": int, "seq": string}
    """
    out = {}
    with open(path, "r") as f:
        current_id = None
        current_pam = None

        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith(">"):
                parts = line[1:].split()
                current_id = parts[0]
                current_pam = int(parts[1])  # SECOND ITEM = PAM INDEX
            else:
                out[current_id] = {
                    "pam": current_pam,
                    "seq": line
                }
                current_id = None
                current_pam = None

    return out


############################################
# LOAD REPAIR TSV → 201-length vector
############################################

def load_repair_outcome_tsv(tsv_path):
    df = pd.read_csv(tsv_path, sep="\t")

    if "Type" not in df.columns:
        raise ValueError(f"{tsv_path} missing 'Type' column!")

    allowed = {"INSERTION", "DELETION"}
    bad = df[~df["Type"].isin(allowed)]
    if len(bad) > 0:
        print("Invalid rows in", tsv_path)
        print(bad)
        raise ValueError("Invalid Type encountered.")

    df["length_change"] = df.apply(
        lambda row: row["Size"] if row["Type"] == "INSERTION" else -row["Size"],
        axis=1
    )

    df = df[(df["length_change"] >= MIN_DELTA) & (df["length_change"] <= MAX_DELTA)]

    grouped = df.groupby("length_change")["countEvents"].sum()

    vec = np.zeros(NUM_CLASSES, dtype=np.float32)
    for delta, count in grouped.items():
        vec[delta + DELTA_OFFSET] = count


    return vec


############################################
# LOAD FORECasT FASTA (TRAIN + TEST)
############################################

train_path = "../data/XCRISP/FORECasT_train.fasta"
test_path  = "../data/XCRISP/FORECasT_test.fasta"

train_dict = load_fasta_id_pam_seq(train_path)
test_dict  = load_fasta_id_pam_seq(test_path)

id_to_info = {**train_dict, **test_dict}

print("Total FORECasT FASTA entries:", len(id_to_info))


############################################
# CELL TYPES (directory names)
############################################

CELL_MAP = {
    "HAP1":          "HAP1",
    "mESC_FORECasT": "mESC_FORECasT",
    "TREX2":         "TREX2",
}


############################################
# BUILD DATASET FOR ONE CELL TYPE
############################################

def build_dataset_for_cell(cell_name):

    cell_dir = os.path.join(FORECAST_DIR, CELL_MAP[cell_name])

    rows = []

    for ID, info in id_to_info.items():
        seq = info["seq"]
        pam_index = info["pam"]

        tsv_path = os.path.join(cell_dir, f"{ID}.tij.sorted.tsv")

        if not os.path.exists(tsv_path):
            #print(f"⚠️ Missing TSV for ID {ID} in cell {cell_name}")
            #print(f"   Expected at: {tsv_path}")
            continue

        y_vec = load_repair_outcome_tsv(tsv_path)

        row = {
            "id": ID,
            "sequence": seq,
            "pam_index": pam_index
        }

        for delta in range(MIN_DELTA, MAX_DELTA + 1):
            row[delta] = y_vec[delta + DELTA_OFFSET]

        rows.append(row)

    return pd.DataFrame(rows)


############################################
# GENERATE ALL FOUR DATASETS
############################################

for cell in CELL_MAP.keys():
    print(f"\n=== Processing FORECasT {cell} ===")
    df = build_dataset_for_cell(cell)
    cell_name = cell.replace("_FORECasT", "")
    out_csv = f"{OUTPUT_PREFIX}_{cell_name}.csv"
    df.to_csv(out_csv, index=False)

    print(f"Saved {cell} dataset at: {out_csv}")
    print(f"Rows: {len(df)}")


Total FORECasT FASTA entries: 11058

=== Processing FORECasT HAP1 ===
Saved HAP1 dataset at: ../data/XCRISP_FORECasT_HAP1.csv
Rows: 10827

=== Processing FORECasT mESC_FORECasT ===
Saved mESC_FORECasT dataset at: ../data/XCRISP_FORECasT_mESC.csv
Rows: 10794

=== Processing FORECasT TREX2 ===
Saved TREX2 dataset at: ../data/XCRISP_FORECasT_TREX2.csv
Rows: 10500


In [5]:

TSV_DIR = "../data/XCRISP/U2OS"
OUTPUT_CSV = "../data/XCRISP_inDelphi_U2OS.csv"


############################################
# FASTA LOADING
############################################

def load_fasta_id_to_seq(path):
    """
    Parses FASTA where each sequence is exactly 1 line and
    >ID OTHER INFO
    SEQUENCE
    """
    id_to_seq = {}
    with open(path, "r") as f:
        current_id = None
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith(">"):
                current_id = line[1:].split()[0]
            else:
                if current_id is None:
                    raise ValueError("Sequence found before header!")
                id_to_seq[current_id] = line
                current_id = None
    return id_to_seq


############################################
# TARGET + PAM DETECTION
############################################

def extract_target_from_id(ID):
    """
    ID looks like: 1_0_2_4_TAGTAACTCAACCTCAGCAG
    Last token after '_' is the target sequence.
    """
    return ID.split("_")[-1]


def find_pam_index(full_seq, target):
    """
    PAM is immediately AFTER the 20bp target.
    Returns index of FIRST PAM base.
    """
    pos = full_seq.find(target)
    if pos == -1:
        raise ValueError(f"Target {target} NOT found in sequence!")
    return pos + len(target)      # PAM begins right after target


############################################
# LOAD REPAIR TSV → 201-length y-vector
############################################

def load_repair_outcome_tsv(tsv_path):
    """
    Loads a single *.tij.sorted.tsv file and returns a 
    vector of length 201 (deltas -100..100).
    """

    df = pd.read_csv(tsv_path, sep="\t")

    # Validate Type column
    if "Type" not in df.columns:
        raise ValueError(f"{tsv_path} missing 'Type' column!")

    allowed = {"INSERTION", "DELETION"}
    bad = df[~df["Type"].isin(allowed)]
    if len(bad) > 0:
        print("Invalid rows in", tsv_path)
        print(bad)
        raise ValueError("Invalid Type encountered.")

    # Compute delta values
    df["length_change"] = df.apply(
        lambda row: row["Size"] if row["Type"] == "INSERTION" else -row["Size"],
        axis=1
    )

    # Filter deltas outside training range
    df = df[(df["length_change"] >= MIN_DELTA) & (df["length_change"] <= MAX_DELTA)]

    # Sum counts per delta
    grouped = df.groupby("length_change")["countEvents"].sum()

    # Build zero vector
    vec = np.zeros(NUM_CLASSES, dtype=np.float32)
    for delta, count in grouped.items():
        vec[delta + DELTA_OFFSET] = count

    return vec


############################################
# LOAD FASTA (TRAIN + TEST)
############################################

id_to_seq_1 = load_fasta_id_to_seq("../data/XCRISP/inDelphi_train.fasta")
id_to_seq_2 = load_fasta_id_to_seq("../data/XCRISP/inDelphi_test.fasta")

id_to_seq = {**id_to_seq_1, **id_to_seq_2}

print("Total FASTA entries loaded:", len(id_to_seq))


############################################
# BUILD FINAL CSV
############################################

rows = []

for ID, seq in id_to_seq.items():

    # TSV path for this target
    tsv_path = os.path.join(TSV_DIR, f"{ID}_indels.tij.sorted.tsv")

    if not os.path.exists(tsv_path):
        print(f"⚠️  WARNING: Missing TSV for ID {ID}")
        continue

    # Load repair outcomes
    y_vec = load_repair_outcome_tsv(tsv_path)

    # Extract target and PAM position
    target = extract_target_from_id(ID)
    pam_index = find_pam_index(seq, target)

    # Build row dictionary
    row = {
        "id": ID,
        "sequence": seq,
        "pam_index": pam_index,
    }

    # Add all outcomes
    for delta in range(MIN_DELTA, MAX_DELTA + 1):
        row[delta] = y_vec[delta + DELTA_OFFSET]

    rows.append(row)

# Convert to DataFrame
df_final = pd.DataFrame(rows)
print("Final dataframe shape:", df_final.shape)

# Save
df_final.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved CSV: {OUTPUT_CSV}")


Total FASTA entries loaded: 1996
⚠️  WARNING: Missing TSV for ID p53_Y163C_hg38_chr17_7675119_to_7675138_(+)_GCTTGCAGATGGCCATGGCG
⚠️  WARNING: Missing TSV for ID overbeek_spacer_84_GGCCACTGTAGTCCTCCAGG
⚠️  WARNING: Missing TSV for ID RS137854557_NF1_NM_000267.3(NF1)_ACTTACAGCTTCTTGTCTCC
⚠️  WARNING: Missing TSV for ID RS754896795_DMD_NM_004006.2(DMD)_GCTTTTCTTCAAGCTGCCCA
⚠️  WARNING: Missing TSV for ID RS121908300_GBA_NM_001005741.2(GBA)_GCCAGACACTTTGTGAAGTA
⚠️  WARNING: Missing TSV for ID 4_0_3_0_CGGCGACGGAGGGCACTCCT
⚠️  WARNING: Missing TSV for ID RS5030804_VHL_NM_000551.3(VHL)_TGCGACTGCAGAAGATGACC
⚠️  WARNING: Missing TSV for ID RS80338732_PRPS1_NM_002764.3(PRPS1)_GCAAATACGCTATCTGTAGC
⚠️  WARNING: Missing TSV for ID libcontrol_Site_A_spcas9_55bp_EMX1_chr2_72933765_to_72933787_(+)_TGCCCCTCCCTCCCTGGCCC
⚠️  WARNING: Missing TSV for ID RS199473074_SCN5A_NM_000335.4(SCN5A)_CTGATACAGTTTTCAGGGCC
⚠️  WARNING: Missing TSV for ID 0_1_4_0_ACCCCACTATTAGTCTATCT
⚠️  WARNING: Missing TSV for ID RS

In [6]:


TSV_DIR = "../data/XCRISP/NHEJdeficient"
OUTPUT_CSV = "../data/XCRISP_inDelphi_mESC_NHEJdeficient.csv"


############################################
# FASTA LOADING
############################################

def load_fasta_id_to_seq(path):
    """
    Parses FASTA where each sequence is exactly 1 line and
    >ID OTHER INFO
    SEQUENCE
    """
    id_to_seq = {}
    with open(path, "r") as f:
        current_id = None
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith(">"):
                current_id = line[1:].split()[0]
            else:
                if current_id is None:
                    raise ValueError("Sequence found before header!")
                id_to_seq[current_id] = line
                current_id = None
    return id_to_seq


############################################
# TARGET + PAM DETECTION
############################################

def extract_target_from_id(ID):
    """
    ID looks like: 1_0_2_4_TAGTAACTCAACCTCAGCAG
    Last token after '_' is the target sequence.
    """
    return ID.split("_")[-1]


def find_pam_index(full_seq, target):
    """
    PAM is immediately AFTER the 20bp target.
    Returns index of FIRST PAM base.
    """
    pos = full_seq.find(target)
    if pos == -1:
        raise ValueError(f"Target {target} NOT found in sequence!")
    return pos + len(target)      # PAM begins right after target


############################################
# LOAD REPAIR TSV → 201-length y-vector
############################################

def load_repair_outcome_tsv(tsv_path):
    """
    Loads a single *.tij.sorted.tsv file and returns a 
    vector of length 201 (deltas -100..100).
    """

    df = pd.read_csv(tsv_path, sep="\t")

    # Validate Type column
    if "Type" not in df.columns:
        raise ValueError(f"{tsv_path} missing 'Type' column!")

    allowed = {"INSERTION", "DELETION"}
    bad = df[~df["Type"].isin(allowed)]
    if len(bad) > 0:
        print("Invalid rows in", tsv_path)
        print(bad)
        raise ValueError("Invalid Type encountered.")

    # Compute delta values
    df["length_change"] = df.apply(
        lambda row: row["Size"] if row["Type"] == "INSERTION" else -row["Size"],
        axis=1
    )

    # Filter deltas outside training range
    df = df[(df["length_change"] >= MIN_DELTA) & (df["length_change"] <= MAX_DELTA)]

    # Sum counts per delta
    grouped = df.groupby("length_change")["countEvents"].sum()

    # Build zero vector
    vec = np.zeros(NUM_CLASSES, dtype=np.float32)
    for delta, count in grouped.items():
        vec[delta + DELTA_OFFSET] = count



    return vec


############################################
# LOAD FASTA (TRAIN + TEST)
############################################

id_to_seq_1 = load_fasta_id_to_seq("../data/XCRISP/inDelphi_train.fasta")
id_to_seq_2 = load_fasta_id_to_seq("../data/XCRISP/inDelphi_test.fasta")

id_to_seq = {**id_to_seq_1, **id_to_seq_2}

print("Total FASTA entries loaded:", len(id_to_seq))


############################################
# BUILD FINAL CSV
############################################

rows = []

for ID, seq in id_to_seq.items():

    # TSV path for this target
    tsv_path = os.path.join(TSV_DIR, f"{ID}_indels.tij.sorted.tsv")

    if not os.path.exists(tsv_path):
        print(f"⚠️  WARNING: Missing TSV for ID {ID}")
        continue

    # Load repair outcomes
    y_vec = load_repair_outcome_tsv(tsv_path)

    # Extract target and PAM position
    target = extract_target_from_id(ID)
    pam_index = find_pam_index(seq, target)

    # Build row dictionary
    row = {
        "id": ID,
        "sequence": seq,
        "pam_index": pam_index,
    }

    # Add all outcomes
    for delta in range(MIN_DELTA, MAX_DELTA + 1):
        row[delta] = y_vec[delta + DELTA_OFFSET]

    rows.append(row)

# Convert to DataFrame
df_final = pd.DataFrame(rows)
print("Final dataframe shape:", df_final.shape)

# Save
df_final.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved CSV: {OUTPUT_CSV}")


Total FASTA entries loaded: 1996
⚠️  WARNING: Missing TSV for ID p53_Y163C_hg38_chr17_7675119_to_7675138_(+)_GCTTGCAGATGGCCATGGCG
⚠️  WARNING: Missing TSV for ID overbeek_spacer_84_GGCCACTGTAGTCCTCCAGG
⚠️  WARNING: Missing TSV for ID RS137854557_NF1_NM_000267.3(NF1)_ACTTACAGCTTCTTGTCTCC
⚠️  WARNING: Missing TSV for ID RS754896795_DMD_NM_004006.2(DMD)_GCTTTTCTTCAAGCTGCCCA
⚠️  WARNING: Missing TSV for ID RS121908300_GBA_NM_001005741.2(GBA)_GCCAGACACTTTGTGAAGTA
⚠️  WARNING: Missing TSV for ID RS5030804_VHL_NM_000551.3(VHL)_TGCGACTGCAGAAGATGACC
⚠️  WARNING: Missing TSV for ID RS80338732_PRPS1_NM_002764.3(PRPS1)_GCAAATACGCTATCTGTAGC
⚠️  WARNING: Missing TSV for ID libcontrol_Site_A_spcas9_55bp_EMX1_chr2_72933765_to_72933787_(+)_TGCCCCTCCCTCCCTGGCCC
⚠️  WARNING: Missing TSV for ID RS199473074_SCN5A_NM_000335.4(SCN5A)_CTGATACAGTTTTCAGGGCC
⚠️  WARNING: Missing TSV for ID 0_0_1_3_TTCCCACTTAGGAACGCATT
⚠️  WARNING: Missing TSV for ID 0_0_0_2_ACCTCCCATACTGGTACTTC
⚠️  WARNING: Missing TSV for ID RS

In [7]:
# open all files and rename pam_index to PAM position
for filename in ['../data/XCRISP_FORECasT_HAP1.csv',
                 '../data/XCRISP_FORECasT_mESC.csv',
                 '../data/XCRISP_FORECasT_TREX2.csv',
                 '../data/XCRISP_inDelphi_mESC.csv',
                 '../data/XCRISP_inDelphi_U2OS.csv',
                 '../data/XCRISP_inDelphi_mESC_NHEJdeficient.csv']:
    path = filename
    df = pd.read_csv(path)
    if "pam_index" in df.columns:
        df = df.rename(columns={"pam_index": "PAM position"})
    df.to_csv(path, index=False)
    print(f"Renamed column in: {path}")


Renamed column in: ../data/XCRISP_FORECasT_HAP1.csv
Renamed column in: ../data/XCRISP_FORECasT_mESC.csv
Renamed column in: ../data/XCRISP_FORECasT_TREX2.csv
Renamed column in: ../data/XCRISP_inDelphi_mESC.csv
Renamed column in: ../data/XCRISP_inDelphi_U2OS.csv
Renamed column in: ../data/XCRISP_inDelphi_mESC_NHEJdeficient.csv
